# 01 – Khảo sát cấu trúc dữ liệu thô (Bước 1)

**Mục tiêu:**
- Khảo sát metadata & cấu trúc dữ liệu thô của **dữ liệu địa từ KAK** (HAPI / IAGA-2002) và **dữ liệu động đất USGS** (CSV / GeoJSON).
- Báo cáo số dòng, số cột, kiểu dữ liệu, thống kê mô tả (min, max, mean, std), khoảng thời gian phủ dữ liệu.
- Phân nhóm cột: Thời gian / Định danh / Đo lường / Chất lượng / Metadata.

**Nguyên tắc:** Chỉ đọc dữ liệu, KHÔNG chỉnh sửa file gốc trong `data/raw/`.


In [1]:
# ── Setup môi trường & nạp thư viện ──────────────────────────────────────────
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Thêm đường dẫn src vào sys.path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import config as cfg
from common.parsers import (
    get_station_meta, load_all_intermagnet,
    load_usgs_csv, load_usgs_geojson
)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DATA_RAW:     {cfg.DATA_RAW}')


PROJECT_ROOT: /home/khang/Code/data-pipeline/geopulse-kak-usgs
DATA_RAW:     /home/khang/Code/data-pipeline/geopulse-kak-usgs/data/raw


## 1. KAK (IAGA-2002 / HAPI) – Đọc Header & Metadata file thô


In [2]:
# Đọc metadata HAPI info của dữ liệu thô KAK
# Mục đích: Đọc thông tin metadata trạm (thành phần đo, sentinel_value, date range)

station_metas = {}
for station in cfg.STATIONS:
    print(f'\n{"─"*60}')
    print(f'Trạm {station} | HAPI Metadata')
    print(f'{"─"*60}')
    meta = get_station_meta(station)
    station_metas[station] = meta
    for k, v in meta.items():
        print(f'  {k:30s}: {v}')



────────────────────────────────────────────────────────────
Trạm KAK | HAPI Metadata
────────────────────────────────────────────────────────────
  station_code                  : KAK
  start_date                    : 2013-01-01T00:00:00Z
  stop_date                     : 2026-04-30T23:59:00Z
  parameters                    : [{'name': 'Time', 'type': 'isotime', 'length': 17, 'units': 'UTC', 'fill': None, 'description': 'The UTC date and time for the data from Kakioka, Japan (KAK)', 'label': 'Date/time'}, {'name': 'Field_Vector', 'type': 'double', 'size': [3], 'units': ['nT', 'nT', 'nT'], 'coordinateSystemName': 'Cartesian', 'vectorComponents': ['x', 'y', 'z'], 'fill': '99999.0', 'description': 'The vector value of the magnetic field at Kakioka, Japan (KAK), for definitions of the vector components see https://intermagnet.org/faq/10.geomagnetic-comp.html', 'label': ['X', 'Y', 'Z']}, {'name': 'Field_Magnitude', 'type': 'double', 'units': 'nT', 'vectorComponents': 'r', 'fill': '99999.0

## 2. KAK (IAGA-2002 / HAPI) – Nạp dữ liệu và báo cáo thống kê


In [3]:
# Nạp toàn bộ dữ liệu chuỗi thời gian địa từ cho trạm KAK
# Dữ liệu thô giữ nguyên giá trị Sentinel (ví dụ 99999.00) để phục vụ bước đếm chất lượng (Bước 2)

station_dfs   = {}  # Lưu DataFrame thô
station_meta2 = {}  # Lưu metadata trạm

for station in cfg.STATIONS:
    try:
        df, meta = load_all_intermagnet(station)
        station_dfs[station]   = df
        station_meta2[station] = meta
        print(f'\n✅ {station}: Nạp xong {len(df):,} dòng')
    except FileNotFoundError as e:
        print(f'⚠  {e}')



✅ KAK: Nạp xong 1,707,840 dòng


In [4]:
# ── Báo cáo cấu trúc DataFrame KAK ───────────────────────────────────────────
for station, df in station_dfs.items():
    meta = station_meta2.get(station, {})
    print(f'\n{"="*65}')
    print(f'  TRẠM {station} – Báo cáo cấu trúc dữ liệu thô')
    print(f'{"="*65}')

    print(f'\n📌 Tổng quan:')
    print(f'   Số dòng  : {len(df):,}')
    print(f'   Số cột   : {len(df.columns)}')
    print(f'   Danh sách: {list(df.columns)}')

    t_min = df["time_utc"].min()
    t_max = df["time_utc"].max()
    print(f'   Từ       : {t_min}')
    print(f'   Đến      : {t_max}')
    print(f'   Thời gian: {t_max - t_min}')

    print(f'\n📌 Kiểu dữ liệu (Data Types):')
    print(df.dtypes.to_string())

    print(f'\n📌 Thống kê mô tả (vẫn còn chứa giá trị Sentinel):')
    print(df.describe(include='all').T.to_string())

    print(f'\n📌 5 dòng dữ liệu đầu tiên:')
    print(df.head().to_string())



  TRẠM KAK – Báo cáo cấu trúc dữ liệu thô

📌 Tổng quan:
   Số dòng  : 1,707,840
   Số cột   : 6
   Danh sách: ['time_utc', 'station', 'x_nt', 'y_nt', 'z_nt', 'f_nt']
   Từ       : 2023-01-01 00:00:00+00:00
   Đến      : 2026-03-31 23:59:00+00:00
   Thời gian: 1185 days 23:59:00

📌 Kiểu dữ liệu (Data Types):
time_utc    datetime64[us, UTC]
station                     str
x_nt                    float64
y_nt                    float64
z_nt                    float64
f_nt                    float64

📌 Thống kê mô tả (vẫn còn chứa giá trị Sentinel):
              count unique  top     freq                       mean                        min                        25%                        50%                        75%                        max   std
time_utc    1707840    NaN  NaN      NaN  2024-08-15 23:59:30+00:00  2023-01-01 00:00:00+00:00  2023-10-24 11:59:45+00:00  2024-08-15 23:59:30+00:00  2025-06-08 11:59:15+00:00  2026-03-31 23:59:00+00:00   NaN
station     1707840      1  K

In [5]:
# ── Phân nhóm các cột trong dữ liệu KAK ──────────────────────────────────────
print('\n=== PHÂN NHÓM CỘT DỮ LIỆU KAK ===')
for station, df in station_dfs.items():
    print(f'\nTrạm {station}:')
    col_groups = {
        'Thời gian' : [c for c in df.columns if 'time' in c.lower() or 'doy' in c.lower()],
        'Định danh' : [c for c in df.columns if c in ('station',)],
        'Đo lường'  : [c for c in df.columns if c.endswith('_nt')],
        'Chất lượng': [c for c in df.columns if 'quality' in c.lower() or 'flag' in c.lower()],
    }
    known = sum([col_groups[k] for k in col_groups], [])
    col_groups['Metadata'] = [c for c in df.columns if c not in known]

    for group, cols in col_groups.items():
        print(f'   {group:12s}: {cols}')



=== PHÂN NHÓM CỘT DỮ LIỆU KAK ===

Trạm KAK:
   Thời gian   : ['time_utc']
   Định danh   : ['station']
   Đo lường    : ['x_nt', 'y_nt', 'z_nt', 'f_nt']
   Chất lượng  : []
   Metadata    : []


## 3. USGS – Nạp và khảo sát dữ liệu CSV động đất


In [6]:
# Nạp dữ liệu thô danh mục động đất USGS từ file CSV
try:
    df_usgs_csv = load_usgs_csv()
    print(f'✅ USGS CSV: Nạp thành công {len(df_usgs_csv):,} sự kiện động đất')
    print(f'   Từ: {df_usgs_csv["time_utc"].min()}')
    print(f'   Đến: {df_usgs_csv["time_utc"].max()}')
    print(f'\n📌 Kiểu dữ liệu (Data Types):')
    print(df_usgs_csv.dtypes.to_string())
    print(f'\n📌 Thống kê mô tả:')
    print(df_usgs_csv.describe(include='all').T.to_string())
    print(f'\n📌 5 dòng dữ liệu đầu tiên:')
    print(df_usgs_csv.head().to_string())
except FileNotFoundError as e:
    print(f'⚠  {e}')
    df_usgs_csv = None


✅ USGS CSV: Nạp thành công 4,184 sự kiện động đất
   Từ: 2023-01-01 21:55:08.183000+00:00
   Đến: 2026-03-31 15:38:47.911000+00:00

📌 Kiểu dữ liệu (Data Types):
time_utc                       datetime64[us, UTC]
latitude                                   float64
longitude                                  float64
depth_km                                   float64
magnitude                                  float64
mag_type                                       str
n_stations                                   int64
azimuthal_gap                                int64
dist_to_nearest_station_deg                float64
rms_residual                               float64
network                                        str
event_id                                       str
updated_utc                    datetime64[us, UTC]
place                                          str
event_type                                     str
horizontal_error_km                        float64
depth_error_km         

## 4. USGS – Nạp và khảo sát dữ liệu GeoJSON động đất


In [7]:
# Nạp dữ liệu thô danh mục động đất USGS từ file GeoJSON và so sánh với CSV
try:
    df_usgs_gj = load_usgs_geojson()
    print(f'✅ USGS GeoJSON: Nạp thành công {len(df_usgs_gj):,} sự kiện động đất')
    print(f'   Từ: {df_usgs_gj["time_utc"].min()}')
    print(f'   Đến: {df_usgs_gj["time_utc"].max()}')
    print(f'\n📌 5 dòng dữ liệu đầu tiên:')
    print(df_usgs_gj.head().to_string())

    # So sánh số lượng sự kiện giữa CSV và GeoJSON
    if df_usgs_csv is not None:
        print(f'\n📌 So sánh CSV ({len(df_usgs_csv):,}) vs GeoJSON ({len(df_usgs_gj):,}):')
        print(f'   Chênh lệch: {len(df_usgs_csv) - len(df_usgs_gj)} sự kiện')
except FileNotFoundError as e:
    print(f'⚠  {e}')
    df_usgs_gj = None


⚠  Không tìm thấy file USGS GeoJSON trong /home/khang/Code/data-pipeline/geopulse-kak-usgs/data/raw/usgs


## 5. Tóm tắt Bước 1 (Khảo sát cấu trúc KAK & USGS)

Tổng hợp lại kết quả khảo sát cấu trúc của 2 nguồn dữ liệu KAK và USGS.


In [8]:
# ── Báo cáo tổng hợp Bước 1 ──────────────────────────────────────────────────
print('=' * 65)
print('  TÓM TẮT BƯỚC 1 – KHẢO SÁT CẤU TRÚC DỮ LIỆU THÔ')
print('=' * 65)

for station in cfg.STATIONS:
    if station not in station_dfs:
        continue
    df   = station_dfs[station]
    meta = station_meta2.get(station, {})
    meas = [c for c in df.columns if c.endswith('_nt')]

    print(f'\n[DỮ LIỆU ĐỊA TỪ KAK / {station}]')
    print(f'  Mã trạm     : {meta.get("station_code")}')
    print(f'  Tọa độ      : {meta.get("geodetic_lat")}°N, {meta.get("geodetic_lon")}°E')
    print(f'  Độ cao      : {meta.get("elevation")} m')
    print(f'  Thành phần  : {meta.get("reported_components")}')
    print(f'  Tần số mẫu  : {meta.get("sample_period_seconds")} giây')
    print(f'  Loại dữ liệu: {meta.get("data_type")}')
    print(f'  Giá trị Lỗi : {meta.get("sentinel_values")} (Sentinel theo chuẩn IAGA-2002)')
    print(f'  Tổng số dòng: {len(df):,}')
    print(f'  Khoảng TG   : {df["time_utc"].min()} → {df["time_utc"].max()}')
    print(f'  Cột đo lường: {meas}')

if df_usgs_csv is not None:
    print(f'\n[DỮ LIỆU ĐỘNG ĐẤT USGS CSV]')
    print(f'  Số sự kiện  : {len(df_usgs_csv):,}')
    print(f'  Khoảng TG   : {df_usgs_csv["time_utc"].min()} → {df_usgs_csv["time_utc"].max()}')
    print(f'  Độ lớn (Mag): {df_usgs_csv["magnitude"].min():.1f} – {df_usgs_csv["magnitude"].max():.1f}')
    print(f'  Danh sách cột: {list(df_usgs_csv.columns)}')

print('\n' + '─'*65)
print('→ Hoàn thành Bước 1. Tiếp tục chuyển sang Bước 2 (Kiểm tra chất lượng).')


  TÓM TẮT BƯỚC 1 – KHẢO SÁT CẤU TRÚC DỮ LIỆU THÔ

[DỮ LIỆU ĐỊA TỪ KAK / KAK]
  Mã trạm     : KAK
  Tọa độ      : None°N, None°E
  Độ cao      : None m
  Thành phần  : XYZF
  Tần số mẫu  : 60 giây
  Loại dữ liệu: quasi-definitive
  Giá trị Lỗi : [99999.0] (Sentinel theo chuẩn IAGA-2002)
  Tổng số dòng: 1,707,840
  Khoảng TG   : 2023-01-01 00:00:00+00:00 → 2026-03-31 23:59:00+00:00
  Cột đo lường: ['x_nt', 'y_nt', 'z_nt', 'f_nt']

[DỮ LIỆU ĐỘNG ĐẤT USGS CSV]
  Số sự kiện  : 4,184
  Khoảng TG   : 2023-01-01 21:55:08.183000+00:00 → 2026-03-31 15:38:47.911000+00:00
  Độ lớn (Mag): 4.0 – 7.6
  Danh sách cột: ['time_utc', 'latitude', 'longitude', 'depth_km', 'magnitude', 'mag_type', 'n_stations', 'azimuthal_gap', 'dist_to_nearest_station_deg', 'rms_residual', 'network', 'event_id', 'updated_utc', 'place', 'event_type', 'horizontal_error_km', 'depth_error_km', 'mag_error', 'n_stations_mag', 'review_status', 'location_source', 'mag_source']

─────────────────────────────────────────────────────